In [61]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import random
import math

from Stock import Coin,User


In [62]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [63]:
class TradeNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(TradeNet, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, output_size)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.relu(x)

        x = self.fc3(x)

        first_part = x[:, :1]
        other_part = x[:, 1:]

        first_activated = torch.relu(first_part)
        other_activated = torch.sigmoid(other_part)

        return torch.cat((first_activated, other_activated), dim=1)


In [64]:
model = TradeNet(128,128,4).to(device)
optimizer = optim.Adam(model.parameters(),lr=0.01)
criterion = nn.MSELoss()

In [65]:
total_params = sum(p.numel() for p in model.parameters())
print(total_params)

33540


In [66]:
import matplotlib.pyplot as plt
import random
from Stock import Coin, User


#--------------------------------------------------------
#функция обвала рынка
def crash(coin_list, percentage):
    for coin in coin_list:
        before = coin.currency
        coin.currency *= (1 - percentage / 100)
        # print(f'{coin.name} before:{before} after:{coin.currency}')


#демонстрация
# d_coin_list = [Coin(f'd_coin{i}',random.randint(10,100),random.randint(10,100)) for i in range(3)]
# crash(d_coin_list,40)

#--------------------------------------------------------
#функция бума на рынке
def stock_boom(coin_list, percentage):
    for coin in coin_list:
        before = coin.currency
        coin.currency *= (1 + percentage / 100)
        # print(f'{coin.name} before:{before} after:{coin.currency}')


#демонстрация
# d_coin_list = [Coin(f'd_coin{i}',random.randint(10,100),random.randint(10,100)) for i in range(3)]
# stock_boom(d_coin_list,10)

#--------------------------------------------------------

def main(steps,users_count,coins_count ):
    percent_to_stock = 4 # процент в биржу

    # создание главного коина
    world_coin = Coin('world_coin', 10 ** 9, 10 ** 9)
    world_manager = [0, [world_coin]]

    users_list = []

    # создание событий
    events = ['crash', 'boom', 'nothing']
    events_weight = [5, 5, 90]

    for i in range(users_count):
        u = User('user' + str(i), 1000, world_coin, world_manager[1])
        u.initialize_nick()
        u.register_in_wc()
        users_list.append(u)

    for i in range(coins_count):
        world_manager[1].append(Coin(f'Coin{i}', random.randint(1000, 100000), random.randint(1000, 100000)))

    history = {
        'prices': {coin.name: [] for coin in world_manager[1]},
        'wealth': {user.nickname: [] for user in users_list}
    }

    for step in range(steps):
        event = random.choices(events, weights=events_weight, k=1)[0]

        if event == 'crash':
            crash(world_manager[1], random.randint(10, 25))
        elif event == 'boom':
            stock_boom(world_manager[1], random.randint(5, 30))

        for user in users_list:
            u_events = ['buy', 'sell']
            target = random.choice(world_manager[1])
            decision = random.choice(u_events)

            if decision == 'buy':
                user.buy(random.randint(1, 1000), target)
            elif decision == 'sell':
                user.sell(random.randint(1, 1000), target)

            user.apply_request(percent_to_stock)

            # запись богатства в историю
            total_wealth = user.wc_count
            for c in world_manager[1]:
                total_wealth += user.wallet.get(c.name, 0) * c.currency
            history['wealth'][user.nickname].append(total_wealth)

        # запись курса монет в историю
        for coin in world_manager[1]:
            history['prices'][coin.name].append(coin.currency)

    return history


#--------------------------------------------------------
# Построение графиков
def graphics(history):
    plt.style.use('ggplot')
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

    # График цен
    for coin_name, prices in history['prices'].items():
        if coin_name != 'world_coin':
            ax1.plot(prices, label=coin_name)
    ax1.set_title('Динамика курсов монет')
    ax1.set_ylabel('Цена в WC')
    ax1.legend()

    # График капитала
    for user_nick, wealth_history in history['wealth'].items():
        ax2.plot(wealth_history, label=user_nick)
    ax2.set_title('Капитализация пользователей (Net Worth)')
    ax2.set_ylabel('Общая стоимость активов')
    ax2.set_xlabel('Шаги симуляции')
    ax2.legend()

    plt.tight_layout()
    plt.show()


#--------------------------------------------------------




In [67]:
def generate_set(steps=128, window_size=16, u_count=4, c_count=4):
    history = main(steps, u_count, c_count)

    all_windows = []
    all_targets = []

    for coin_name, prices in history['prices'].items():
        if coin_name == 'world_coin':
            continue

        for i in range(len(prices) - window_size):
            window = prices[i : i + window_size]
            all_windows.append(window)

            target = calc_single_y(window)
            all_targets.append(target)

    return torch.tensor(all_windows, dtype=torch.float32), torch.tensor(all_targets, dtype=torch.float32)

def calc_single_y(price_block):
    k = (max(price_block) - min(price_block)) / len(price_block)
    if k == 0: k = 1 # Защита от деления на 0


    y_last = price_block[-1]
    x = 1 / (y_last * k) if y_last != 0 else 0

    # Нормализация
    v_min, v_max = min(price_block), max(price_block)
    n = (y_last - v_min) / (v_max - v_min) if v_max != v_min else 0.5

    return [x, 1-n,n, 0.5]

X_train, Y_train = generate_set(steps=8192, window_size=128)

Nickname has been initialized as @user0
User: @user0 registered in system
Nickname has been initialized as @user1
User: @user1 registered in system
Nickname has been initialized as @user2
User: @user2 registered in system
Nickname has been initialized as @user3
User: @user3 registered in system


In [68]:
epochs = 3000
batch_size = 512

loss_history = []

for epoch in range(epochs):
    model.to(device)
    model.train()

    indices = torch.randperm(X_train.size(0))

    epoch_loss = 0

    for i in range(0, len(X_train), batch_size):
        batch_indices = indices[i : i + batch_size]
        batch_x = X_train[batch_indices].to(device)
        batch_y = Y_train[batch_indices].to(device)

        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / (len(X_train) / batch_size)
    loss_history.append(avg_loss)

    if (epoch + 1) % 100 == 0:
        print(f'Epoch [{epoch+1}/{epochs}] | Loss: {avg_loss:.6f}')


        # torch.save(model.state_dict(), f'nanokoin{epoch+1}.pth')

plt.figure(figsize=(10, 5))
plt.plot(loss_history, label='Training Loss', color='royalblue', linewidth=2)
plt.title('Процесс обучения нейросети (Loss Curve)')
plt.xlabel('Эпоха')
plt.ylabel('MSE Loss')
plt.yscale('log')
plt.grid(True, which="both", ls="-", alpha=0.5)
plt.legend()
plt.show()

Epoch [100/3000] | Loss: 31653300151826346147840.000000
Epoch [200/3000] | Loss: 31166822235988423606272.000000


KeyboardInterrupt: 

In [ ]:
test = torch.tensor([[i for i in range(400,528)]],dtype=torch.float32).to(device)
model.eval()
with torch.no_grad():
    output = model(test)
    print(output)

In [ ]:
device = torch.device('cpu')
model.to(device)
torch.save(model.state_dict(), 'megakoinT2.pth')

In [ ]:
#manykoin 16 32 4
#lowkoin 16 32 4
#nanokoin 16 64 4
#megakoin 128 128 4
#megakoinT2 128 128 4